# Day 3 - Problem Set

## Problem 1 - Plotting data
### Background
One of the most important things a program can do in scientific computing is the reduction and plotting of data, nearly all plotting in Python is done using the `matplotlib` library, which is extremely versatile, but also has a bit of a learning curve, for the work this session we are going to cover the more commonly used methods of producing plots using this library.

### Task
In this folder there are a number of CSV (comma-separated values) files containing reduced data from the [MESA MIST stellar evolution project](https://waps.cfa.harvard.edu/MIST/). Write a program that can:

- Read in one of these files.
- Produce a luminosity vs. time plot using `matplotlib`.

Afterwards:

- Modify the program to read in _all_ the `.csv` files in the folder, and include all the resultant lines in the same plot.
- Add labels and change the axes scale to be more appropriate.
- Write a program to produce a HR diagram from the same data files.

### Bonus
- Write a version of this program that makes multiple plots into the same image, the luminosity vs. time graph and the HR diagram.

### Hints
- The first line of the `.csv` files is a header containing what each column represents, `numpy` should ignore this column when importing.
- Nearly all Python programs in physics call in the plotting routines of `matplotlib` using the phrase `impoart matplotlib.pyplot as plt`, why do you think this would be?
- `matplotlib` is sometimes quite hard to get around unless you know what you are looking for, I'd highly recommend [checking on the documentation](https://matplotlib.org/stable/index.html).
- You can control the scale of a plots x and y axis with the `plt.xscale()` and `plt.yscale()` functions, for shorthand there are `semilog` and `loglog` versions of `plt.plot()`.
- Reading a CSV file into an array will produce a 2-dimensional array, the array itself has to be sliced into its individual columns in order to display properly.

## Problem 2 - Fitting data
A very important aspect of scientific computing is estimating values from data, one of the most common ways to do this is with the `scipy` `curve_fit()` functions, where you can define an equation you want to fit data to, and the function will estimate the value by optimizing, by changing initial guess values to reduce the uncertainty.

### Task

A damped oscillator can be described with the following differential equation:

$$
\frac{d^2 x}{dt^2} + \gamma \frac{dx}{dt} + \omega_0^2 x = 0,
$$

where $x$ is the oscillator position, $t$ is the time, $\gamma$ is the coefficient of resistance, and $\omega_0$ is the natural frequency. Solving this equation for $x$, we can describe the position over time such that:

$$
x(t) = A e^{-\gamma t / 2} \cos{\left( \omega_d t + \delta \right)} , 
$$

where $A$ is the amplitude, $\delta$ is a phase constant and $\omega_d$ is the damped frequency:

$$
\omega_d^2 = \left( \omega_0^2 - \frac{\gamma^2}{4} \right).
$$

Using these equations, the following program has produces $x$ and $t$ data of a damped oscillator with randomized $\gamma$ and $\omega_0$ values, and written them to the file `oscillator.csv`. A small amount of random noise has also been added in order to simulate instrument uncertainty.

In [207]:
# Generate Damped Oscillator Data
import numpy as np
from random import uniform # random.uniform function is used to generate random float values between two values
# Derive the parameters of the damped oscillator
gamma = uniform(0.25, 2.0) # Random damping coefficient between 0.25 and 1.0
omega_0 = uniform(1, 10) # Random natural frequency between 1 and 10
print(f"Generated fundamental parameters: gamma={gamma:.3f}, omega_0={omega_0:.3f}, xi={gamma/omega_0:.3f}") # Print the generated parameters, including the damping ratio (xi)
if gamma >= omega_0: # Check if the system is overdamped or critically damped
  raise ValueError("Warning: The system is overdamped or critically damped (xi < 1). Please generate new parameters.")
omega_d_2 = (omega_0**2 - (gamma**2 / 4)) # Calculate the square of the damped frequency
omega_d = np.sqrt(omega_d_2) # Calculate the damped frequency (omega_d) using the formula: omega_d = sqrt(omega_0^2 - gamma^2)
A = uniform(0.25, 1.0) # Random amplitude between 0.25 and 1.0
delta = uniform(-np.pi, np.pi) # Random phase shift between -pi and pi

t = np.linspace(0, 10, 1000) # Generate time data from 0 to 10 seconds, 1000 points are used to create a smooth curve 
x = A * np.exp(-gamma * t / 2) * np.cos(omega_d * t + delta) # Calculate the position data using the formula: x(t) = A * exp(-gamma/2 * t) * cos(omega_d * t + delta)
x += np.random.normal(0, 0.025, size=t.shape) # Add random noise to the position data to simulate measurement errors

data = np.column_stack((t, x)) # Combine resultant data into a single array
np.savetxt("oscillator.csv", data, delimiter=",", header="time,position", fmt="%.5e") # Save the data to a CSV file

print(f"Generated Damped Oscillator data with A={A:.3f}, gamma={gamma:.3f}, omega_0={omega_0:.3f}, delta={delta:.3f}")


Generated fundamental parameters: gamma=1.605, omega_0=7.305, xi=0.220
Generated Damped Oscillator data with A=0.891, gamma=1.605, omega_0=7.305, delta=-2.469


Produce a program that will:

- Read and plot the harmonic oscillator data produced by the above program.
- Perform a fit to estimate the parameters of the data's sinusoidal curve, in particular $\omega_d$ and $\gamma$.
- Plot a fitted line using those parameters.
- Plot additional lines of $Ae^{\gamma t/2}$ and $-Ae^{\gamma t/2}$ to show the amplitude decay of the system (this will also help show that your fit is accurate!).
- Print the parameter estimates in a way that is legible to the user.
- Calculate $\omega_0$ from your estimated values of $\omega_d$ and $\gamma$.
- Add a title to the plot with $\gamma$ and $\omega_0$.

Here is an example of the plot using a different dataset:

![Example of the damped oscillator output](oscillator_example.png)
> *Example of the damped oscillator output.*

### Bonus
- Calculate the uncertainty of the fitting estimates as well, and print those with the parameter estimates.
- Propagate the uncertainty to find $\delta \omega_0$.

### Hints
- This problem is a lot harder than the last few problems, but the next one isn't too bad, I promise.
- Look, you could just uncomment the diagnostic print statements I left in, but where is the fun in that?
- The documentation on curve fitting is in [`scipy.optimize`](https://docs.scipy.org/doc/scipy/reference/optimize.html).
- Calculating the uncertainties of a covariant matrix is performed by diagonalizing it, and then taking the square root, this can be done using [`numpy`](https://numpy.org/devdocs/reference/generated/numpy.diag.html).
- Propagating uncertainties can be done manually, but you can use a [Python library for that](https://pythonhosted.org/uncertainties/).

In [ ]:
# Damped Oscillator Data Analysis Code



## Problem 3 - Plotting images
### Background
In addition to table based data, producing image plots is very important for scientific work. `matplotlib` can be used to plot images as well as lines and points, and can also be used to decode basic images.
However, most image data in astronomy is stored in the form of `FITS` files, which can contain multiple images, tables and information in the same file. These are not supported by `matplotlib`, but can be converted into data that `matplotlib` can read using a library such as `astropy`.

### Task
#### Part 1
- Write a program that can read and plot an image file with the filename provided by the user (I have provided a test image, `sipi.tiff`)

#### Part 2 
- Make a program that will read in the Hubble Space Telescope data (`hst_11137_01_wfpc2_total_wf_drz.fits`) in this folder using the `astropy` FITS functions.
- Add to the program a way to plot the image data using `matplotlib`.
  - The image data may require scaling to view properly, with a minimum brightness of the 1st percentile of the image and the maximum brightness at the 99th percentile.
- Add the correct WCS coordinates to the image.

### Hints
- This one will require referring back to the `astropy` documentation quite a bit!
- As I've said before, complex problems like calculating the WCS coordinates of an image are usually already done by someone else, and are usually in the form of a library.


## Problem 5 - Programming outside of CoLab
### Background
While we've been using CoLab for everything in these sessions, this is largely because setting up Python on a computer can be difficult and this is a very short course, in the lab you will probably be using Jupyter notebooks and Python programs running off of your own computer on the one given to you.